# GPU03 — خ۸ (F08): سلسله‌مراتبی و بیزی — NUTS روی GPU

> بند 7.17 `doc/WBS-phase7-modeling.md` · اسپرینت C، ردیف «خ۸ بیزی سلسله‌مراتبی».

**چرا این خانواده منطبق‌ترین با ساختار مسئله است:** F10 (ICC روز=۰.۲۲۵ و
سلف=۰.۲۰۵ ⇒ اثرات تصادفی **متقاطع**)، F59 (۸۳٪ واریانس = شوک مشترک روز)، F07
(بیش‌پراکندگی صعودی ⇒ Beta-Binomial نه دوجمله‌ای).

⭐ **و یک مزیت که هیچ خانواده‌ی دیگری ندارد:** روز آزمون یک روز **جدید** است، پس
اثر تصادفی‌اش معلوم نیست و باید از پیشین پسین‌آموخته قرعه بخورد. مدل‌های نقطه‌ای
عملاً وانمود می‌کنند «شوک روز آینده صفر است» — یعنی همان چیزی که ۸۳٪ واریانس را
می‌سازد از عدم‌قطعیت حذف می‌شود. بند 7.17 انتظار دارد این خانواده حتی اگر pinball
را نبرد، **کالیبراسیون** بهتری بدهد — و بند ۶.۴ کالیبراسیون را معیار اصلی می‌داند.

| model_id | عضو WBS | ساختار |
|---|---|---|
| `bhm_beta_binomial_restaurant` | ۱/۵ | فقط اثر تصادفی سلف |
| `bhm_beta_binomial_crossed` | ⭐ ۲/۵ | روز + سلف متقاطع |
| `bhm_varying_dispersion` | ⭐ ۶ | $\phi = f(\log Res)$ — پاسخ به F06/F07 |

**بودجه‌ی هدف: ~۹۰ دقیقه.**

## سلول ۱ — نصب وابستگی‌ها

`torch`/`jax` روی کولب و کگل از پیش نصب‌اند و نسخه‌شان با درایور CUDA همان ماشین
هماهنگ است؛ نصب دوباره‌شان چند گیگابایت دانلود و گاهی ناسازگاری درایور می‌آورد.
پس فقط چیزهایی نصب می‌شوند که واقعاً نیستند. نسخه‌ی دقیق هرچه استفاده شد در سلول ۵
چاپ و در MLflow ثبت می‌شود (بازتولیدپذیری از راه **ثبت**، نه پین‌کردن).
فهرست کامل: `requirements-gpu.txt` داخل همین بسته.

In [ ]:
!pip install -q numpyro arviz optuna mlflow tabulate

## سلول ۲ — بارگذاری بسته‌ی کد + داده

⚠️ **کد اصلی داخل نوت‌بوک نوشته نمی‌شود** (بند 7.8.4، قاعده‌ی «`notebooks/` = روایت،
`src/` = حقیقت»). این نوت‌بوک فقط `src/` را import و روایت می‌کند.

`gpu_bundle.zip` را با `python -m src.models.gpu_bundle` بسازید و در Drive بگذارید
(یا در کگل به‌عنوان Dataset آپلود کنید). داخلش: کل `src/`، چهار فایل
`data/processed/` که سلول ۳ رویشان assert می‌زند، و نتایج CPU خانواده‌های قبلی برای
جدول مقایسه.

In [ ]:
MODE = "colab"          # ← "colab" یا "kaggle"
BUNDLE_COLAB  = "/content/drive/MyDrive/phase7/gpu_bundle.zip"   # ← مسیر خودتان
BUNDLE_KAGGLE = "/kaggle/input/phase7-bundle/gpu_bundle.zip"

import os, sys, zipfile, pathlib

if MODE == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    bundle, workdir = BUNDLE_COLAB, pathlib.Path("/content/phase7")
else:
    bundle, workdir = BUNDLE_KAGGLE, pathlib.Path("/kaggle/working/phase7")

workdir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    z.extractall(workdir)
os.chdir(workdir)
sys.path.insert(0, str(workdir))
print("محتوای بسته:", sorted(p.name for p in workdir.iterdir()))

## سلول ۳ — ⭐ دروازه‌ی انصاف A1 (بند 7.7.3)

**اگر هش‌ها نخوانند، نوت‌بوک همین‌جا می‌ایستد.** بدون این assert هیچ اثباتی وجود
ندارد که این اجرا روی همان foldها و همان snapshot دادهٔ خانواده‌های CPU انجام شده —
و هر run با `cv_folds_hash` نامنطبق از جدول مقایسه‌ی فاز ۷ حذف می‌شود. مقادیر زیر
از بخش «قفل فاز ۷» `doc/data_manifest.md` آمده‌اند.

In [ ]:
from src.models.gpu_runner import assert_fairness_gate

EXPECTED_CV_FOLDS_HASH      = "bd08d6f7c801ee0611121e404774251de07a480ac1589b12eb7c64f8044b78d4"
EXPECTED_DATA_SNAPSHOT_HASH = "68b4cb8517d292599b2f161f779758b9f3254d60302849f39d81650d0bd9fba0"   # data/processed/features_A_v1.parquet

from src.models.gpu_runner import load_l1
data = load_l1()

assert_fairness_gate(data, EXPECTED_CV_FOLDS_HASH, EXPECTED_DATA_SNAPSHOT_HASH)
print(data.summary())

## سلول ۴ — بذر تصادفی سراسری

`set_global_seed()` تنها منبع بذر پروژه است (`AGENTS.md`). قطعیت کامل روی GPU
تضمین‌شدنی نیست — به‌همین‌دلیل قاعده‌ی **سه seed** (A7، بند 7.16.3) در مرحله‌ی
قهرمان اجرا می‌شود و پراکندگی بین seedها خودش گزارش می‌گردد، نه پنهان.

In [ ]:
from src.config import set_global_seed
from src.models.gpu_runner import setup_torch_determinism

set_global_seed()
setup_torch_determinism(strict=False)   # strict=True بعضی op های cuDNN را می‌شکند

## سلول ۵ — سخت‌افزار

زمان‌های اجرا فقط با دانستن سخت‌افزار قابل تفسیرند (بند 7.8.2).

In [ ]:
!nvidia-smi

from src.models.gpu_runner import device_report

DEVICE = device_report()
DEVICE

## سلول ۶ — ردیابی MLflow جدا

`mlruns_gpu/` جداست تا ادغام با `mlruns/` محلی (بند 7.8.3 گام ۵) امن و قابل بازگشت
باشد. tag اجباری `compute` هم همین‌جا ست می‌شود.

In [ ]:
from pathlib import Path
from src.models.gpu_runner import use_gpu_tracking

COMPUTE = "colab"     # اگر روی کگل اجرا می‌کنید: "kaggle"
print("MLflow →", use_gpu_tracking("mlruns_gpu"))

# reports/gpu/ را همین‌جا می‌سازیم — سلول‌های ۷-ب/۷-ج (PPC، حساسیت پیشین) مستقیم
# CSV آن‌جا می‌نویسند، پیش از آنکه save_family_report/package_outputs بسازدش
Path("reports/gpu").mkdir(parents=True, exist_ok=True)


## سلول ۷-الف — R0 + چک‌لیست همگرایی بند 7.17.3

برای مدل بیزی، «شاهد همگرایی» قاعده‌ی A6 جایش را به چک‌لیست بند 7.17.3 می‌دهد:
R̂ < ۱.۰۱ · ESS > ۴۰۰ · صفر divergence. `convergence_report` هر سه را عدد می‌دهد،
نه بررسی چشمی نمودار.

In [ ]:
import pandas as pd
from src.models.families import f08_bayesian as fam
from src.models.gpu_runner import smoke_test
from src.models.axes import TUNING_TAU

MODEL_IDS = ["bhm_beta_binomial_restaurant", "bhm_beta_binomial_crossed",
             "bhm_varying_dispersion"]

smoke, diagnostics = [], []
tr0, te0 = data.folds[0]
for mid in MODEL_IDS:
    smoke.append(smoke_test(fam.FITTERS[mid], data,
                            hyperparams={"num_warmup": 300, "num_samples": 300}))
    model = fam.FITTERS[mid].fit(tr0, TUNING_TAU, num_warmup=300, num_samples=300)
    diagnostics.append({"مدل": mid, **model.diagnostics})

convergence_table = pd.DataFrame(diagnostics)
convergence_table

## سلول ۷-ب — ⭐ Posterior Predictive Check (بند 7.17.3، مهم‌ترین بند چک‌لیست)

> «مدلی که میانگین را درست می‌زند ولی چولگی را بازتولید نمی‌کند، کوانتایل‌هایش غلط
> است.» هدف: چولگی ~۴.۰۶ (F02) و تورم صفر ~۴.۹٪ (F03).

In [ ]:
ppc_rows = []
for mid in MODEL_IDS:
    model = fam.FITTERS[mid].fit(tr0, TUNING_TAU, num_warmup=300, num_samples=300)
    ppc_rows.append({"مدل": mid, **fam.posterior_predictive_check(model, te0)})
ppc_table = pd.DataFrame(ppc_rows)
ppc_table.to_csv("reports/gpu/F08_ppc.csv", index=False)
ppc_table

## سلول ۷-ج — تحلیل حساسیت پیشین (اجباری، بند 7.17.2)

سه مجموعه پیشین (`tight` / `default` / `wide`) روی fold۰. اگر نتیجه به پیشین حساس
باشد، هر ادعایی درباره‌ی این خانواده باید با آن حساسیت گزارش شود.

In [ ]:
from src.baselines import operational_metrics

prior_rows = []
for preset in fam.PRIOR_PRESETS:
    m = fam.FITTERS["bhm_beta_binomial_crossed"].fit(
        tr0, TUNING_TAU, prior_preset=preset, num_warmup=300, num_samples=300)
    pred = m.predict(te0, TUNING_TAU)
    met = operational_metrics(te0, pred, TUNING_TAU)
    prior_rows.append({"پیشین": preset, **fam.PRIOR_PRESETS[preset],
                       "pinball": round(met["pinball"], 5),
                       "پوشش": round(met["coverage"], 4),
                       "max_R_hat": round(m.diagnostics["max_r_hat"], 4),
                       "divergences": m.diagnostics["n_divergences"]})
prior_table = pd.DataFrame(prior_rows)
prior_table.to_csv("reports/gpu/F08_prior_sensitivity.csv", index=False)
prior_table

## سلول ۷-د — R2: تنظیم با بودجه‌ی زمانی

⚠️ فضای این خانواده عمداً **کاردینالیتی محدود** (۲۷) اعلام شده — بدون آن، جدول
بودجه ۶۰ trial می‌داد که با NUTS یعنی چند ده ساعت. دقیقاً همان اشتباهی که یافته‌ی ۱۲
مستند کرده است.

In [ ]:
from src.models.gpu_runner import run_gpu_study
from src.models.spaces import SPACES

BUDGET_MINUTES = {"bhm_beta_binomial_restaurant": 12,
                  "bhm_beta_binomial_crossed": 22,
                  "bhm_varying_dispersion": 22}

studies = [run_gpu_study(fam.FITTERS[mid], SPACES[mid].fn, data,
                         family=fam.FAMILY, feature_set=fam.FEATURE_SET,
                         budget_minutes=BUDGET_MINUTES[mid], compute=COMPUTE, seed=42)
           for mid in MODEL_IDS]

## سلول ۷-ه — قهرمان + ACI + DM

⭐ نکته‌ی مورد انتظار: ستون «پوشش» این خانواده. بند 7.17.5 می‌گوید مقایسه‌ی پوشش
کوانتایل پسین با مدل‌های غیربیزی «جایی است که این خانواده احتمالاً می‌برد» — حتی
اگر pinball را نبرد.

In [ ]:
from src.models.gpu_runner import finalize_champion

best = min(studies, key=lambda s: s.best_pinball)
print(f"قهرمان: {best.model_id} (pinball={best.best_pinball:.5f})\n")
champions = [finalize_champion(fam.FITTERS[best.model_id], data, best,
                               feature_set=fam.FEATURE_SET, seeds=(42, 1234),
                               compute=COMPUTE, run_aci=True)]

## سلول ۷-و — راستی‌آزمایی مدل ذخیره‌شده

مدل بیزی وزن ندارد، **توزیع پسین** دارد — همان چیزی که در `.npz` ذخیره شده. با
همین فایل می‌شود بدون هیچ نمونه‌گیری دوباره، برای هر داده‌ی جدید و **هر τ** پیش‌بینی
گرفت. سلول زیر همین را نشان می‌دهد.

In [ ]:
import numpy as np
from pathlib import Path
from src.models.axes import TAU_GRID

stem = Path(f"models/gpu/F08/{best.model_id}/{best.model_id}__s42__fold0")
reloaded = fam.FITTERS[best.model_id].load(stem)
print("τ | پیش‌بینی میانگین (از همان پسین ذخیره‌شده، بدون نمونه‌گیری دوباره)")
for t in TAU_GRID:
    print(f"{t:.2f} | {reloaded.predict(data.folds[0][1], t).mean():.5f}")
print("\nتشخیص همگرایی ذخیره‌شده:", reloaded.diagnostics)
print("✅ مدل ذخیره‌شده قابل استفاده است")

## سلول ۷-ز — گزارش فارسی کامل

In [ ]:
from src.models.gpu_runner import render_family_report, save_family_report

notes = [
    "چک‌لیست همگرایی بند 7.17.3 جای قاعده‌ی A6 را می‌گیرد — جدول `F08_ppc.csv` و ستون‌های R̂/ESS.",
    "اثر روزِ آزمون از پیشین قرعه می‌خورد نه از پسین — این تفاوت کالیبراسیون بیزی با مدل نقطه‌ای است.",
    "تحلیل حساسیت پیشین اجباری اجرا شد: `reports/gpu/F08_prior_sensitivity.csv`.",
    "پارامترسازی غیرمرکزی (z~N(0,1); u=σ·z) — بند 7.17.2، بدون آن NUTS در قیف نیل واگرا می‌شود.",
]
report = render_family_report("F08", "خ۸ — سلسله‌مراتبی و بیزی (NUTS روی GPU)",
                              studies, champions, smoke, DEVICE, notes)
report += ("\n## چک‌لیست همگرایی (بند 7.17.3)\n\n"
           + convergence_table.to_markdown(index=False)
           + "\n\n## Posterior Predictive Check\n\n" + ppc_table.to_markdown(index=False)
           + "\n\n## حساسیت پیشین\n\n" + prior_table.to_markdown(index=False) + "\n")
save_family_report("F08", report, "F08_bayesian_L1")
print(report)

## سلول ۸ — بسته‌بندی خروجی (تکه‌های ۱۰۰ مگابایتی)

همه‌ی خروجی‌ها — `mlruns_gpu/` (هر trial + قهرمان‌ها با artifact مدل)،
`models/gpu/` (وزن‌ها و پیش‌پردازش هر fold/seed)، `reports/gpu/` (JSON و گزارش
فارسی)، و `optuna_studies/*.db` (تا اجرای بعدی از همین‌جا ادامه دهد) — در یک zip
جمع و به تکه‌های ۱۰۰ مگابایتی شکسته می‌شوند. هر تکه SHA-256 خودش را در
`MANIFEST_F08_bayesian_L1.json` دارد، پس اگر دانلود یکی خراب شد فقط همان یکی دوباره گرفته
می‌شود.

In [ ]:
from src.models.gpu_runner import package_outputs, download_parts

manifest = package_outputs(tag="F08_bayesian_L1", part_mb=100)
download_parts()      # روی کولب دانلود می‌کند؛ روی کگل فایل‌ها در خروجی session می‌مانند

---
## پس از اجرا — چرخه‌ی بازگشت (بند 7.8.3)

```
۱. این نوت‌بوک اجراشده (File → Download .ipynb، با تمام خروجی‌ها) →
   notebooks/gpu/executed/{name}__{تاریخ}.ipynb    ← حتی اگر آزمایش شکست خورد؛ شکست هم داده است
۲. تکه‌ها → ریشه‌ی مخزن:  cat gpu_outputs_*.zip.part* > gpu_outputs.zip && unzip gpu_outputs.zip
۳. rsync -a mlruns_gpu/ mlruns/        (ادغام MLflow)
۴. کارت مدل ۱۴ گامی → reports/models/{model_id}.md
۵. make mlflow-ui  →  runهای جدید با tag compute=colab باید دیده شوند
```